# Time Series Models

**PyBroker v2** introduces support for backtesting time series models. Instead of training on examples individually, these models make predictions based on a series' own past values.

To show how this works, we will backtest two different strategies. The first relies on a volatility forecast from a [GARCH(1,1)](https://en.wikipedia.org/wiki/Autoregressive_conditional_heteroskedasticity) model built with the [arch](https://arch.readthedocs.io/) library. The second strategy uses a rolling regression that is refit on every bar. Since **PyBroker** does not include `arch` by default, you must install it first by running `pip install arch`.

In [1]:
import arch
import numpy as np
import pandas as pd
import pybroker
from pybroker import Strategy, YFinance

pybroker.enable_data_source_cache("time_series")

## Forecasting Volatility with GARCH

GARCH will model the volatility of a return series, so we start by defining an [indicator](https://www.pybroker.com/en/latest/reference/pybroker.indicator.html#pybroker.indicator.indicator) for log returns:

In [2]:
def log_return(bar_data):
    close = bar_data.close
    ret = np.full_like(close, np.nan)
    ret[1:] = np.diff(np.log(close))
    return ret


log_return_ind = pybroker.indicator("log_return", log_return)

The training function scales the log returns to percentages for more reliable estimation:

In [3]:
def train_garch(symbol, train_data, test_data):
    returns = (
        pd.concat((train_data["log_return"], test_data["log_return"]))
        .dropna()
        .to_numpy()
        * 100
    )
    n_train = int(train_data["log_return"].count())
    # Estimate on the train window only; the test returns are held out
    # for forecasting.
    am = arch.arch_model(returns, vol="GARCH", p=1, q=1)
    return am.fit(last_obs=n_train, disp="off")

The model is built using the combined returns from both the train and test windows. Passing `last_obs` ensures that parameter estimation relies solely on the train window and isolates the test returns to prevent data leakage.

For every test data bar, the prediction function uses the trained model to forecast the variance of the next bar:

In [4]:
def predict_garch(model, data):
    # Position of the current bar in the model's return series.
    pos = model.fit_stop + len(data) - 1
    # Forecast the next bar's variance from the trained model.
    forecast = model.forecast(horizon=1, start=pos)
    variance = forecast.variance.to_numpy()[0, 0]
    # Annualize the one-day volatility forecast.
    return np.sqrt(variance) / 100 * np.sqrt(252)

Because the model already stores the full return series, the model input is the location of the current bar. By doing this, the forecast then only uses returns up to that point.

By default, **PyBroker** passes all test window data to the model in a single call to make a make a prediction. While this vectorized approach is efficient, it fails for autoregressive models since they rely on the previous step's output to generate their next forecast. Passing `per_bar=True` to [pybroker.model(...)](https://www.pybroker.com/en/latest/reference/pybroker.model.html#pybroker.model.model) will cause the [predict_fn](https://www.pybroker.com/en/latest/reference/pybroker.model.html#pybroker.model.model) to be called once per bar of input:

In [5]:
garch_model = pybroker.model(
    "garch",
    train_garch,
    predict_fn=predict_garch,
    indicators=[log_return_ind],
    per_bar=True,
)

The strategy then uses the volatility forecast from [ctx.preds](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.preds) as a regime filter. It enters a long position when forecast volatility is below the threshold, and exits when it rises above:

In [6]:
VOL_THRESHOLD = 0.30


def vol_filter(ctx):
    pred_vol = ctx.preds("garch")[-1]
    if not ctx.long_pos():
        # Enter while forecast volatility is below the threshold.
        if pred_vol < VOL_THRESHOLD:
            ctx.buy_shares = ctx.calc_target_shares(0.5)
    elif pred_vol > VOL_THRESHOLD:
        # Exit when forecast volatility rises above the threshold.
        ctx.sell_all_shares()


strategy = Strategy(YFinance(), start_date="1/1/2021", end_date="1/1/2026")
strategy.add_execution(vol_filter, ["SBUX", "IBM"], models=garch_model)
result = strategy.walkforward(windows=2, train_size=0.5)
result.metrics_df.head(20)

Backtesting: 2021-01-01 00:00:00 to 2026-01-01 00:00:00



Loading bar data...


[                       0%                       ]

[*********************100%***********************]  2 of 2 completed

Loaded bar data: 0:00:00 



...

 86% (361 of 418) |###################   | Elapsed Time: 0:00:00 ETA:   0:00:00

100% (418 of 418) |######################| Elapsed Time: 0:00:00 ETA:  00:00:00

100% (418 of 418) |######################| Elapsed Time: 0:00:00 Time:  0:00:00

Finished backtest: 0:00:02


,name,value
0,trade_count,6
1,initial_market_value,100000.0
2,end_market_value,159861.28
3,total_pnl,-9971.31
4,unrealized_pnl,69832.59
5,total_return_pct,-9.97131
6,total_profit,7728.96
7,total_loss,-17700.27
8,total_fees,0.0
9,max_drawdown,-35133.94


## Random Forest on Lagged Returns

The second strategy trains a [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) to predict the next bar's return from lagged returns: 

In [7]:
from sklearn.ensemble import RandomForestRegressor


def train_forest(symbol, train_data, test_data, lag_train, lag_test):
    rets = train_data["log_return"].to_numpy()
    # Regress each next-bar return on the bar's return and its lags.
    forest = RandomForestRegressor(random_state=42)
    forest.fit(lag_train[:-1], rets[1:])
    return forest

The training function receives `lag_train` and `lag_test` parameters built from configuring the model with the desired number of [lags](https://www.pybroker.com/en/latest/reference/pybroker.model.html#pybroker.model.model). Each parameter is a feature matrix containing one row per example. These rows begin with the bar's `log_return` value, followed by its lagged return values.

Unlike the per-bar GARCH model, the `predict_fn` uses **PyBroker's** default behavior and passes the entire test window in a single  call to generate the model's predictions:

In [8]:
def predict_forest(model, data):
    return model.predict(data)

Registering the model with [lags](https://www.pybroker.com/en/latest/reference/pybroker.model.html#pybroker.model.model) set to `3` will include the past three lagged values for each column declared in [lag_cols](https://www.pybroker.com/en/latest/reference/pybroker.model.html#pybroker.model.model):

In [9]:
forest_model = pybroker.model(
    "forest",
    train_forest,
    predict_fn=predict_forest,
    lags=3,
    lag_cols=[log_return_ind],
)

The strategy buys when [ctx.preds](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.preds) for the next-bar return is positive and exits when it is negative. We then run a [walkforward](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy.walkforward) backtest:

In [10]:
def trade_forest(ctx):
    pred = ctx.preds("forest")[-1]
    if not ctx.long_pos():
        if pred > 0:
            ctx.buy_shares = ctx.calc_target_shares(0.5)
    elif pred < 0:
        ctx.sell_all_shares()


strategy.clear_executions()
strategy.add_execution(trade_forest, ["SBUX", "IBM"], models=forest_model)
result = strategy.walkforward(windows=2, train_size=0.5)
result.metrics_df.head(20)

Backtesting: 2021-01-01 00:00:00 to 2026-01-01 00:00:00



Loaded cached bar data.



Computing indicators...


  0% (0 of 2) |                          | Elapsed Time: 0:00:00 ETA:  --:--:--

100% (2 of 2) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00

...

Test split: 2024-05-02 00:00:00 to 2025-12-31 00:00:00

  0% (0 of 418) |                        | Elapsed Time: 0:00:00 ETA:  --:--:--

100% (418 of 418) |######################| Elapsed Time: 0:00:00 Time:  0:00:00

Finished backtest: 0:00:02


,name,value
0,trade_count,413
1,initial_market_value,100000.0
2,end_market_value,126876.85
3,total_pnl,27205.09
4,unrealized_pnl,-328.24
5,total_return_pct,27.20509
6,total_profit,187008.87
7,total_loss,-159803.78
8,total_fees,0.0
9,max_drawdown,-18211.41
